# ASD Screening Agent — Upgraded (v2)

**Upgrades from v1:**
- ✅ Upgrade 1: Persistent conversation memory via LangGraph `MemorySaver`
- ✅ Upgrade 2: Trait extraction — identifies which of 14 behavioural traits are present
- ✅ Upgrade 3: Personalised guidance agent — context-aware follow-up conversation
- ✅ Upgrade 4: Severity/concern level scoring (High / Moderate / Monitor)
- ✅ New FSM stage: `guidance` — persistent post-result conversation loop
- ✅ State extended: carries `last_assessment` dict for cross-turn context

## Cell 1 — Install Libraries

In [ ]:
!pip install langchain langgraph langchain-groq transformers torch scikit-learn xgboost gdown

## Cell 2 — API Key

In [ ]:
from google.colab import userdata
groq_api_key = userdata.get('GROQ_API_KEY')

## Cell 3 — Imports

In [ ]:
import re, os, json, joblib, zipfile
import gdown, torch, numpy as np, pandas as pd

from typing import Annotated, Literal, Optional
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver   # Upgrade 1: persistent memory

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print('All imports successful.')

## Cell 4 — LLM

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-120b", groq_api_key=groq_api_key)

## Cell 5 — Load ML Models

In [ ]:
os.makedirs('/content/model', exist_ok=True)
url = 'https://drive.google.com/drive/folders/1zNUQbmaNz-UhS0Mkxi8XcuclOd4PuxFJ?usp=drive_link'
gdown.download_folder(url, output='/content/model', quiet=False)

with zipfile.ZipFile('/content/model/asd_classifier_model.zip', 'r') as z:
    z.extractall('/content/model/')

xgboost_model = joblib.load('/content/model/xgboost_asd_model.pkl')

with open('/content/model/feature_cols.json') as _f:
    FEATURE_COLS = json.load(_f)
print('XGBoost loaded — features:', FEATURE_COLS)

BERT_MAX_LENGTH = 64
tokenizer  = AutoTokenizer.from_pretrained('/content/model/asd_classifier_model')
bert_model = AutoModelForSequenceClassification.from_pretrained('/content/model/asd_classifier_model')
bert_model.eval()
print('BERT loaded — MAX_LENGTH:', BERT_MAX_LENGTH)

## Cell 6 — Trait Taxonomy & Severity Map

Upgrade 2 & 4: defines the 14 behavioural traits from the BERT training dataset
and assigns each a concern level used for severity scoring.

In [ ]:
# ── 14 behavioural traits from Mendeley ASD Behavioural Text dataset ──────────
# These are the exact Class labels the BERT model was trained on.
# Used for LLM-based trait extraction grounded in real clinical taxonomy.

TRAIT_TAXONOMY = [
    "Eye Contact",           # reduced or absent eye contact
    "Attention Response",    # not responding to name when called
    "Follow Pointing",       # not following pointing gestures
    "Word Repetition",       # echolalia, repeating words or phrases
    "Repetitive Behavior",   # repetitive movements or actions
    "Focused Attention",     # intense focus on specific objects or topics
    "Emotional Empathy",     # limited response to others' emotions
    "Sharing Interest",      # not pointing to share interest with others
    "Sign Communication",    # limited use of gestures
    "Change Reaction",       # strong resistance to routine changes
    "Noise Sensitivity",     # distress caused by sounds
    "Finger Movements",      # unusual hand or finger movements
    "Toy Arranging",         # lining up or arranging objects repetitively
    "Tiptoe Flapping",       # walking on tiptoes or hand flapping
]

# ── Upgrade 4: Severity mapping ────────────────────────────────────────────────
# Clinical prioritisation based on ASD diagnostic criteria (DSM-5 domains).
# High = core social-communication markers
# Moderate = restricted/repetitive behaviour markers
# Monitor = sensory/motor markers (common but less specific)

HIGH_CONCERN = {
    "Eye Contact", "Attention Response", "Follow Pointing",
    "Word Repetition", "Emotional Empathy", "Sharing Interest"
}
MODERATE_CONCERN = {
    "Repetitive Behavior", "Change Reaction", "Sign Communication",
    "Focused Attention"
}
MONITOR_CONCERN = {
    "Noise Sensitivity", "Finger Movements", "Toy Arranging", "Tiptoe Flapping"
}

def compute_concern_level(traits_present: list) -> dict:
    """
    Upgrade 4: Compute overall concern level from detected traits.
    Returns a dict with level, counts, and rationale string.
    """
    high    = [t for t in traits_present if t in HIGH_CONCERN]
    moderate = [t for t in traits_present if t in MODERATE_CONCERN]
    monitor  = [t for t in traits_present if t in MONITOR_CONCERN]

    if len(high) >= 2:
        level = "High"
        rationale = f"{len(high)} high-concern social-communication traits detected."
    elif len(high) == 1 or len(moderate) >= 2:
        level = "Moderate"
        rationale = f"{len(high)} high-concern and {len(moderate)} moderate-concern traits detected."
    elif len(traits_present) > 0:
        level = "Monitor"
        rationale = f"{len(traits_present)} trait(s) noted, primarily sensory/motor markers."
    else:
        level = "Low"
        rationale = "No specific behavioural traits identified in the description."

    return {
        "level": level,
        "rationale": rationale,
        "high": high,
        "moderate": moderate,
        "monitor": monitor,
    }

print('Trait taxonomy and severity map defined.')
print(f'Total traits: {len(TRAIT_TAXONOMY)}')
print(f'High concern: {len(HIGH_CONCERN)} | Moderate: {len(MODERATE_CONCERN)} | Monitor: {len(MONITOR_CONCERN)}')

## Cell 7 — State Definition

Extended state now carries `last_assessment` for cross-turn guidance context.
New FSM stage `guidance` added for persistent post-result conversation.

In [ ]:
# ── FSM stages ─────────────────────────────────────────────────────────────────
#   idle                 -> greet, ask yes/no
#   choose_method        -> ask questionnaire or text
#   awaiting_answers     -> Q-CHAT-10 shown, waiting for 11 answers
#   awaiting_description -> text prompt shown, waiting for description
#   guidance             -> NEW: post-result conversational loop
#                           user can ask about traits, meaning, next steps
#                           exits when user says 'another' or 'exit'

class State(TypedDict):
    messages       : Annotated[list, add_messages]  # full conversation history
    answer         : str                             # final answer to display
    stage          : str                             # current FSM stage
    last_assessment: Optional[dict]                  # Upgrade 2/3/4: stores last result context

print('State defined with last_assessment field.')

## Cell 8 — Core Prediction & Utility Functions

Same XGBoost and BERT inference as v1. `parse_answers` and `is_valid_description` unchanged.

In [ ]:
def questionnaire_predict(features: list) -> tuple:
    """
    Run XGBoost on 11 Q-CHAT-10 answers.
    Returns (result_string, label, confidence, proba_asd, proba_non)
    so upstream callers can use the structured values.
    """
    sample_df = pd.DataFrame([features], columns=FEATURE_COLS)
    proba     = xgboost_model.predict_proba(sample_df)[0].astype(float)  # fix: numpy.float32 → Python float
    pred      = int(np.argmax(proba))
    label     = 'ASD' if pred == 1 else 'Non-ASD'
    confidence = float(proba[pred] * 100)

    result_str = (
        f"Questionnaire Result\n"
        f"{'─'*40}\n"
        f"Assessment  : {label}\n"
        f"Confidence  : {confidence:.1f}%  "
        f"(ASD: {proba[1]*100:.1f}% | Non-ASD: {proba[0]*100:.1f}%)\n"
        f"Features    : {dict(zip(FEATURE_COLS, features))}\n\n"
        f"DISCLAIMER: This is a screening tool only, not a medical diagnosis.\n"
        f"Always consult a qualified healthcare professional."
    )
    return result_str, label, float(confidence), float(proba[1]*100), float(proba[0]*100)


def text_predict(description: str) -> tuple:
    """
    Run BERT on a free-text behaviour description.
    Returns (result_string, label, confidence, proba_asd, proba_non)
    """
    inputs = tokenizer(description, return_tensors='pt',
                       truncation=True, padding=True, max_length=BERT_MAX_LENGTH)
    with torch.no_grad():
        logits = bert_model(**inputs).logits
        proba  = torch.softmax(logits, dim=1)[0]
        pred   = torch.argmax(proba).item()

    label      = 'ASD' if pred == 1 else 'Non-ASD'
    confidence = proba[pred].item() * 100
    preview    = description[:200] + ('...' if len(description) > 200 else '')

    result_str = (
        f"Text Analysis Result\n"
        f"{'─'*40}\n"
        f"Assessment  : {label}\n"
        f"Confidence  : {confidence:.1f}%  "
        f"(ASD: {proba[1].item()*100:.1f}% | Non-ASD: {proba[0].item()*100:.1f}%)\n"
        f"Input text  : \"{preview}\"\n\n"
        f"DISCLAIMER: This is a screening tool only, not a medical diagnosis.\n"
        f"Always consult a qualified healthcare professional."
    )
    return result_str, label, confidence, proba[1].item()*100, proba[0].item()*100


def parse_answers(raw: str):
    """Validate and parse comma-separated Q-CHAT-10 answers. Unchanged from v1."""
    try:
        values = [int(x.strip()) for x in raw.split(',')]
    except ValueError:
        return 'Please use only 0 or 1 separated by commas. Example: 0,1,0,1,1,0,0,0,1,0,0'
    if len(values) != 11:
        return f'I need exactly 11 values but got {len(values)}. Please re-enter all 11 answers.'
    bad = [v for v in values if v not in (0, 1)]
    if bad:
        return f'All values must be 0 or 1. Found: {bad}. Please re-enter.'
    return values


def is_valid_description(text: str) -> bool:
    """Relevance gate: LLM yes/no check. Unchanged from v1."""
    check = llm.invoke([
        SystemMessage(content=(
            "You are a strict input filter for a medical screening tool.\n"
            "Reply ONLY with yes or no — nothing else.\n"
            "Question: Is the following text a description of a child's behaviour, "
            "development, communication, or social traits?"
        )),
        HumanMessage(content=text)
    ])
    return check.content.strip().lower().startswith('yes')


print('Prediction and utility functions defined.')

## Cell 9 — Trait Extraction Function (Upgrade 2)

After BERT inference, this calls the LLM to identify which specific behavioural
traits from the 14-category taxonomy are present in the description.
The LLM cannot invent traits — it can only identify from the fixed taxonomy list.

In [ ]:
TRAIT_EXTRACTION_SYSTEM = """
You are a clinical behavioural analysis assistant for ASD screening.
You will be given a parent's description of their child's behaviour.

Your task is to identify which of the following 14 behavioural traits
are present, absent, or uncertain based ONLY on what is stated or clearly
implied in the text. Do NOT infer traits that are not mentioned.

The 14 traits are:
1. Eye Contact — reduced or absent eye contact
2. Attention Response — not responding to name when called
3. Follow Pointing — not following pointing gestures
4. Word Repetition — echolalia, repeating words or phrases
5. Repetitive Behavior — repetitive movements or actions
6. Focused Attention — intense focus on specific objects or topics
7. Emotional Empathy — limited response to others' emotions
8. Sharing Interest — not pointing to share interest with others
9. Sign Communication — limited use of gestures
10. Change Reaction — strong resistance to routine changes
11. Noise Sensitivity — distress caused by sounds
12. Finger Movements — unusual hand or finger movements
13. Toy Arranging — lining up or arranging objects repetitively
14. Tiptoe Flapping — walking on tiptoes or hand flapping

Reply ONLY with valid JSON. No other text.
Format:
{"traits_present": ["Trait Name", ...],
 "traits_absent": ["Trait Name", ...],
 "traits_uncertain": ["Trait Name", ...]}
"""


def extract_traits(description: str) -> dict:
    """
    Upgrade 2: Extract behavioural traits from a free-text description.
    Calls LLM with fixed taxonomy — cannot hallucinate new traits.
    Returns dict with traits_present, traits_absent, traits_uncertain.
    Falls back to empty lists on parse failure.
    """
    try:
        raw = llm.invoke([
            SystemMessage(content=TRAIT_EXTRACTION_SYSTEM),
            HumanMessage(content=f"Description: {description}")
        ])
        text = raw.content.strip().strip('`')
        if text.startswith('json'):
            text = text[4:].strip()
        # Extract first JSON block robustly
        match = re.search(r'\{.*\}', text, re.DOTALL)
        text  = match.group(0) if match else text
        data  = json.loads(text)

        # Validate: only allow traits from taxonomy
        valid = set(TRAIT_TAXONOMY)
        return {
            'traits_present'  : [t for t in data.get('traits_present',   []) if t in valid],
            'traits_absent'   : [t for t in data.get('traits_absent',    []) if t in valid],
            'traits_uncertain': [t for t in data.get('traits_uncertain', []) if t in valid],
        }
    except Exception as e:
        print(f'Trait extraction failed: {e}')
        return {'traits_present': [], 'traits_absent': [], 'traits_uncertain': []}


def format_trait_report(trait_result: dict, concern: dict) -> str:
    """
    Format trait extraction and severity results into a human-readable block
    appended to the prediction result.
    """
    present   = trait_result.get('traits_present', [])
    uncertain = trait_result.get('traits_uncertain', [])

    lines = []
    lines.append(f"\n{'─'*40}")
    lines.append("Behavioural Trait Analysis")
    lines.append(f"{'─'*40}")

    if present:
        lines.append(f"Traits detected ({len(present)}):")
        for t in present:
            if t in HIGH_CONCERN:
                tag = "[High concern]"
            elif t in MODERATE_CONCERN:
                tag = "[Moderate concern]"
            else:
                tag = "[Monitor]"
            lines.append(f"  • {t} {tag}")
    else:
        lines.append("Traits detected: None clearly identified in description.")

    if uncertain:
        lines.append(f"Uncertain (need more detail): {', '.join(uncertain)}")

    lines.append(f"\nOverall concern level : {concern['level']}")
    lines.append(f"Rationale             : {concern['rationale']}")
    lines.append(f"\nYou can now ask me about any of these traits, what they mean,")
    lines.append(f"what to monitor at home, or when to seek professional help.")
    lines.append(f"Type 'another' to run a new assessment or 'exit' to end.")

    return '\n'.join(lines)


print('Trait extraction functions defined.')

## Cell 10 — Agent Definitions

Four agents: supervisor, questionnaire_agent, text_agent, guidance_agent (new).
Supervisor updated to handle new `guidance` stage.

In [ ]:
# ── SUPERVISOR SYSTEM PROMPT ──────────────────────────────────────────────────
# Updated to include guidance stage routing.

SUPERVISOR_SYSTEM = """
You are the supervisor of an ASD (Autism Spectrum Disorder) screening assistant.
Your ONLY domain is ASD screening in children. Politely refuse all off-topic questions.

You will receive the current conversation stage and must reply with ONLY valid JSON:
{"next": "<routing>", "reply": "<message to user>"}

Routing values:
  "questionnaire_agent" -> user wants the Q-CHAT-10 questionnaire method
  "text_agent"          -> user wants the text description method
  "guidance_agent"      -> user is in post-result and asking a follow-up question
  "end"                 -> no sub-agent needed, just reply

Stage rules:
  idle                -> Greet warmly, explain you do ASD behavioural trait screening, ask yes/no. next=end
  idle + yes/sure/ok  -> Ask them to choose: questionnaire or text description. next=end
  idle + no           -> Acknowledge, say you are available. next=end
  choose_method + questionnaire/qchat/1 -> next=questionnaire_agent
  choose_method + text/describe/2       -> next=text_agent
  guidance + another/new/again          -> Ask them to choose method again. next=end (stage->choose_method)
  guidance + exit/bye/done/quit         -> Thank them, say goodbye. next=end (stage->idle)
  guidance + any question about results -> next=guidance_agent
  Any stage + hi/hello                  -> Greet and introduce yourself. next=end
  Any stage + who are you               -> Explain purpose: ASD screening + trait detection + guidance. next=end
  Any stage + off-topic                 -> Politely decline, redirect. next=end
  Any stage + bye/exit/quit             -> Say goodbye. next=end
"""


def supervisor_agent(state: State) -> State:
    """
    Supervisor: reads stage, routes via LLM JSON decision.
    Fast-path bypasses LLM when stage is awaiting_answers/awaiting_description.
    Updated to handle guidance stage routing.
    """
    print("--- Supervisor Agent ---")
    stage    = state.get('stage', 'idle')
    messages = state.get('messages', [])

    # Fast-path: bypass LLM entirely for data-collection stages
    if stage == 'awaiting_answers':
        state['answer'] = ''
        return state
    if stage == 'awaiting_description':
        state['answer'] = ''
        return state

    # Build history for LLM
    history = [
        SystemMessage(content=SUPERVISOR_SYSTEM),
        SystemMessage(content=f'Current stage: {stage}'),
    ]
    for msg in messages:
        if isinstance(msg, (HumanMessage, AIMessage)):
            history.append(msg)
    if not messages:
        history.append(HumanMessage(content='[START]'))

    raw = llm.invoke(history)

    # Robust JSON parse with fallback
    try:
        text = raw.content.strip().strip('`')
        if text.startswith('json'):
            text = text[4:].strip()
        match = re.search(r'\{.*\}', text, re.DOTALL)
        text  = match.group(0) if match else text
        data  = json.loads(text)
        next_ = data.get('next', 'end')
        reply = data.get('reply', '')
    except (json.JSONDecodeError, AttributeError):
        next_ = 'end'
        reply = raw.content.strip()

    valid_routes = ('questionnaire_agent', 'text_agent', 'guidance_agent', 'end')
    if next_ not in valid_routes:
        next_ = 'end'

    # Update FSM stage
    if next_ == 'questionnaire_agent':
        state['stage'] = 'awaiting_answers'
    elif next_ == 'text_agent':
        state['stage'] = 'awaiting_description'
    elif next_ == 'guidance_agent':
        state['stage'] = 'guidance'  # stays in guidance
    elif stage == 'idle' and next_ == 'end':
        low = reply.lower()
        if any(w in low for w in ['questionnaire', 'text', 'method', 'choose', 'which']):
            state['stage'] = 'choose_method'
    elif stage == 'guidance' and next_ == 'end':
        low = reply.lower()
        if any(w in low for w in ['questionnaire', 'text', 'method', 'choose', 'which']):
            state['stage'] = 'choose_method'
        elif any(w in low for w in ['goodbye', 'bye', 'take care', 'available']):
            state['stage'] = 'idle'

    state['messages'] = messages + [AIMessage(content=reply)]
    state['answer']   = reply if next_ == 'end' else ''
    print(f'decision: {next_}')
    return state


# ─────────────────────────────────────────────────────────────────────────────
# QUESTIONNAIRE AGENT — unchanged logic, updated to return tuple from predict
# ─────────────────────────────────────────────────────────────────────────────

def questionnaire_agent(state: State) -> State:
    """
    Handles Q-CHAT-10 questionnaire flow.
    On result: stores assessment in last_assessment, moves to guidance stage.
    """
    print("--- Questionnaire Agent ---")
    messages = state.get('messages', [])

    last_human = next(
        (m.content.strip() for m in reversed(messages) if isinstance(m, HumanMessage)),
        None
    )

    QUESTIONS = (
        "Q-CHAT-10 Questionnaire\n\n"
        "Answer 0 (No / Female) or 1 (Yes / Male) for each question:\n\n"
        " 1. [A1]  Does your child look at you when you call his/her name?\n"
        " 2. [A2]  How easy is it to get eye contact with your child?\n"
        " 3. [A3]  Does your child point to indicate they want something?\n"
        " 4. [A4]  Does your child point to share interest with you?\n"
        " 5. [A5]  Does your child pretend? (e.g. care for dolls, toy phone)\n"
        " 6. [A6]  Does your child follow where you're looking?\n"
        " 7. [A7]  If someone is upset, does your child try to comfort them?\n"
        " 8. [A8]  Would you describe your child's first words as normal?\n"
        " 9. [A9]  Does your child use simple gestures? (e.g. wave goodbye)\n"
        "10. [A10] Does your child stare at nothing with no apparent purpose?\n"
        "11. [Sex] Child biological sex (0=Female, 1=Male)\n\n"
        "Enter all 11 answers as comma-separated numbers in order A1→A10→Sex.\n"
        "Example: 0,1,0,1,1,0,0,0,1,0,0\n\n"
        "Your answers:"
    )

    if state.get('stage') == 'awaiting_answers' and last_human and ',' in last_human:
        result = parse_answers(last_human)
        if isinstance(result, str):
            reply = result + '\n\n' + QUESTIONS
            state['messages'] = messages + [AIMessage(content=reply)]
            state['answer']   = reply
            state['stage']    = 'awaiting_answers'
        else:
            # Run XGBoost
            result_str, label, conf, p_asd, p_non = questionnaire_predict(result)

            # Upgrade 2 & 4: trait extraction not available from questionnaire
            # (no free text to analyse) — note this clearly for the user
            trait_note = (
                "\n\n─────────────────────────────\n"
                "Note: Specific behavioural trait analysis is available when using\n"
                "the text description method. The questionnaire provides a structured\n"
                "screening score only.\n"
                "\nYou can now ask me what your result means, what to do next,\n"
                "or type 'another' for a new assessment / 'exit' to end."
            )

            answer = result_str + trait_note

            # Store in last_assessment for guidance agent
            state['last_assessment'] = {
                'method'        : 'questionnaire',
                'label'         : label,
                'confidence'    : conf,
                'proba_asd'     : p_asd,
                'proba_non'     : p_non,
                'traits_present': [],
                'traits_absent' : [],
                'concern_level' : 'N/A — questionnaire method',
                'features'      : dict(zip(FEATURE_COLS, result)),
            }

            state['messages'] = messages + [AIMessage(content=answer)]
            state['answer']   = answer
            state['stage']    = 'guidance'  # enter guidance stage
        return state

    # First visit — show questions
    state['messages'] = messages + [AIMessage(content=QUESTIONS)]
    state['answer']   = QUESTIONS
    state['stage']    = 'awaiting_answers'
    return state


# ─────────────────────────────────────────────────────────────────────────────
# TEXT AGENT — adds trait extraction + severity scoring after BERT inference
# ─────────────────────────────────────────────────────────────────────────────

def text_agent(state: State) -> State:
    """
    Handles free-text description flow.
    After BERT: runs trait extraction (Upgrade 2) and severity scoring (Upgrade 4).
    Stores full assessment in last_assessment, moves to guidance stage.
    """
    print("--- Text Agent ---")
    messages = state.get('messages', [])

    last_human = next(
        (m.content.strip() for m in reversed(messages) if isinstance(m, HumanMessage)),
        None
    )

    PROMPT = (
        "Text Description Method\n\n"
        "Describe the child's behaviour in your own words.\n"
        "Include details about social interactions, communication,\n"
        "repetitive behaviours, and response to surroundings.\n\n"
        "Example: 'My 3-year-old rarely makes eye contact and does not respond to his name.'\n\n"
        "Your description:"
    )

    if state.get('stage') == 'awaiting_description' and last_human and len(last_human.split()) >= 3:

        # Relevance gate
        print("--- Relevance Gate Check ---")
        if not is_valid_description(last_human):
            reply = (
                "That doesn't look like a behavioural description.\n\n"
                "Please describe the child's behaviour, communication, or social traits.\n"
                "Example: 'My 3-year-old rarely makes eye contact and does not respond to his name.'\n\n"
                "Your description:"
            )
            state['messages'] = messages + [AIMessage(content=reply)]
            state['answer']   = reply
            state['stage']    = 'awaiting_description'
            return state

        # BERT inference
        result_str, label, conf, p_asd, p_non = text_predict(last_human)

        # Upgrade 2: trait extraction
        print("--- Trait Extraction ---")
        trait_result = extract_traits(last_human)

        # Upgrade 4: severity scoring
        concern = compute_concern_level(trait_result['traits_present'])

        # Format full report
        trait_report = format_trait_report(trait_result, concern)
        answer = result_str + trait_report

        # Store complete assessment context for guidance agent
        state['last_assessment'] = {
            'method'          : 'text',
            'label'           : label,
            'confidence'      : conf,
            'proba_asd'       : p_asd,
            'proba_non'       : p_non,
            'description'     : last_human,
            'traits_present'  : trait_result['traits_present'],
            'traits_absent'   : trait_result['traits_absent'],
            'traits_uncertain': trait_result['traits_uncertain'],
            'concern_level'   : concern['level'],
            'concern_rationale': concern['rationale'],
            'high_traits'     : concern['high'],
            'moderate_traits' : concern['moderate'],
            'monitor_traits'  : concern['monitor'],
        }

        state['messages'] = messages + [AIMessage(content=answer)]
        state['answer']   = answer
        state['stage']    = 'guidance'  # enter guidance stage
        return state

    # First visit — show prompt
    state['messages'] = messages + [AIMessage(content=PROMPT)]
    state['answer']   = PROMPT
    state['stage']    = 'awaiting_description'
    return state


# ─────────────────────────────────────────────────────────────────────────────
# GUIDANCE AGENT — Upgrade 3: personalised follow-up conversation
# ─────────────────────────────────────────────────────────────────────────────

GUIDANCE_SYSTEM = """
You are a compassionate ASD screening support assistant.
A screening assessment has just been completed. You have full context of the results.

Your role is to:
1. Answer the parent's questions about the screening result
2. Explain what each identified trait means in simple, parent-friendly language
3. Suggest practical things to observe or try at home
4. Explain when and how to seek a professional developmental assessment
5. Provide emotional support — be warm, reassuring, and non-alarmist

You MUST:
- NEVER upgrade the screening result to a diagnosis
- ALWAYS recommend professional consultation for any concerning result
- ONLY discuss traits that were identified in THIS assessment
- Stay factually grounded — do not speculate beyond what the data shows
- Be conversational, not a list — answer like a knowledgeable, kind advisor
- If asked about something outside ASD, politely redirect

Assessment context:
{context}
"""


def guidance_agent(state: State) -> State:
    """
    Upgrade 3: Personalised post-result guidance conversation.
    Has full access to last_assessment context.
    Handles follow-up questions about traits, next steps, meaning of results.
    Memory-aware: passes full conversation history to LLM.
    """
    print("--- Guidance Agent ---")
    messages        = state.get('messages', [])
    last_assessment = state.get('last_assessment', {})

    # Build readable context string from last_assessment
    ctx_lines = []
    if last_assessment:
        ctx_lines.append(f"Method        : {last_assessment.get('method', 'unknown')}")
        ctx_lines.append(f"Assessment    : {last_assessment.get('label', 'unknown')}")
        ctx_lines.append(f"Confidence    : {last_assessment.get('confidence', 0):.1f}%")
        ctx_lines.append(f"Concern level : {last_assessment.get('concern_level', 'N/A')}")
        traits = last_assessment.get('traits_present', [])
        if traits:
            ctx_lines.append(f"Traits found  : {', '.join(traits)}")
            ctx_lines.append(f"High concern  : {', '.join(last_assessment.get('high_traits', [])) or 'None'}")
            ctx_lines.append(f"Moderate      : {', '.join(last_assessment.get('moderate_traits', [])) or 'None'}")
        else:
            ctx_lines.append("Traits found  : None specifically identified")
        if last_assessment.get('description'):
            ctx_lines.append(f"Description   : {last_assessment['description'][:300]}")
    else:
        ctx_lines.append("No assessment context available.")

    context_str = '\n'.join(ctx_lines)
    system_with_context = GUIDANCE_SYSTEM.format(context=context_str)

    # Build history — include full conversation for memory
    history = [SystemMessage(content=system_with_context)]
    for msg in messages:
        if isinstance(msg, (HumanMessage, AIMessage)):
            history.append(msg)

    response = llm.invoke(history)
    reply    = response.content.strip()

    state['messages'] = messages + [AIMessage(content=reply)]
    state['answer']   = reply
    state['stage']    = 'guidance'  # stay in guidance until user exits
    return state


print('All four agents defined.')

## Cell 11 — Routing Logic

Updated to include `guidance_agent` as a valid routing target.

In [ ]:
def routing_logic(state: State) -> Literal["questionnaire_agent", "text_agent", "guidance_agent", "end"]:
    """
    Maps FSM stage to the correct next node.
    Updated to route guidance stage to guidance_agent.
    """
    stage = state.get('stage', 'idle')

    if stage == 'awaiting_answers':
        decision = 'questionnaire_agent'
    elif stage == 'awaiting_description':
        decision = 'text_agent'
    elif stage == 'guidance':
        decision = 'guidance_agent'
    else:
        decision = 'end'

    print(f'routing → {decision}')
    return decision

print('Routing logic defined.')

## Cell 12 — Build Graph with MemorySaver (Upgrade 1)

Four nodes, MemorySaver checkpointer for persistent cross-turn memory.

In [ ]:
# Upgrade 1: MemorySaver gives the graph persistent memory across invocations.
# Each conversation session gets a unique thread_id.
# The full state (messages, stage, last_assessment) is saved and restored
# automatically between turns — no manual state passing needed.

memory = MemorySaver()

graph_workflow = StateGraph(State)

graph_workflow.add_node("supervisor_agent",    supervisor_agent)
graph_workflow.add_node("questionnaire_agent", questionnaire_agent)
graph_workflow.add_node("text_agent",          text_agent)
graph_workflow.add_node("guidance_agent",       guidance_agent)   # new

graph_workflow.add_edge(START, "supervisor_agent")

graph_workflow.add_conditional_edges(
    "supervisor_agent",
    routing_logic,
    {
        "questionnaire_agent": "questionnaire_agent",
        "text_agent"         : "text_agent",
        "guidance_agent"     : "guidance_agent",
        "end"                : END,
    }
)

graph_workflow.add_edge("questionnaire_agent", END)
graph_workflow.add_edge("text_agent",          END)
graph_workflow.add_edge("guidance_agent",      END)

# Compile with MemorySaver
agents = graph_workflow.compile(checkpointer=memory)

print('Graph compiled with MemorySaver.')
print('Nodes:', ['supervisor_agent', 'questionnaire_agent', 'text_agent', 'guidance_agent'])

## Cell 13 — Visualise Graph

In [ ]:
from IPython.display import Image, display
display(Image(agents.get_graph().draw_mermaid_png()))

## Cell 14 — Conversation Loop

Uses `thread_id` config for MemorySaver — every session gets a unique ID.
The agent now remembers the full conversation across all turns.

In [ ]:
import uuid

# Each session gets a unique thread_id.
# MemorySaver uses this to store and restore full state between invocations.
# Change thread_id to start a completely fresh conversation.
session_id = str(uuid.uuid4())
config     = {"configurable": {"thread_id": session_id}}

print(f"Session ID: {session_id}")
print("─" * 50)

# Initial state — only needed for the very first invocation.
# After that, MemorySaver restores state automatically from the checkpoint.
initial_state = {
    "messages"       : [],
    "answer"         : "",
    "stage"          : "idle",
    "last_assessment": None,
}

# Opening greeting
result = agents.invoke(initial_state, config=config)
print(f"ASD Agent: {result['answer']}\n")

while True:
    try:
        user_input = input("You (type 'exit' or 'quit' to stop): ").strip()
        if user_input.lower() in ['exit', 'quit']:
            print("Exiting session.")
            break
        if not user_input:
            continue

        # MemorySaver handles state — just pass the new user message.
        # No need to manually carry state between turns.
        result = agents.invoke(
            {"messages": [HumanMessage(content=user_input)]},
            config=config
        )
        print(f"ASD Agent: {result['answer']}\n")

    except KeyboardInterrupt:
        print("\nSession interrupted.")
        break
    except Exception as e:
        print(f"Error: {str(e)}\n")
        continue